# Semana 2: leer NanoAOD con uproot y awkward (`01_intro_uproot.ipynb`)

Programa: [`docs/programa_mes1.md`](../../docs/programa_mes1.md), sección *Semana 2*.

**Muestra:** archivo educativo de CMS Open Data `Run2012BC_DoubleMuParked_Muons.root`
(colisiones pp a √s = 8 TeV de 2012, *dataset* DoubleMuParked, unos 61.5 millones de eventos, solo muones).

**Cómo trabajar este notebook.** Completa las celdas marcadas con `# TODO`. Las celdas de **verificación**
(las que terminan en `assert`) ya están escritas: no las borres ni cambies las tolerancias. Si una verificación
falla, intenta corregirlo tú; si sigue fallando, anótalo en `bitacora.md` en *qué no entendí*. Las celdas
*Tu respuesta* se contestan con tus palabras (2 a 5 frases) y se discuten en la reunión.

Antes del commit: *Kernel → Restart & Run All* (todo debe correr de principio a fin) y luego borra las salidas.

In [ ]:
# --- Configuración: funciona igual en Google Colab y en un entorno conda local ---
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RREYESAL/Hands_on_SSIPN"  # <- si trabajas en un fork, pon aquí su URL
DIR_COLAB = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "uproot", "awkward", "vector", "hist", "mplhep", "pyyaml", "aiohttp", "requests", "psutil"], check=True)
    if not DIR_COLAB.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(DIR_COLAB)], check=True)
    os.chdir(DIR_COLAB / "notebooks" / "mes1")

# Buscar la raíz del repositorio (la carpeta que contiene src/hepss) y hacer importable hepss
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hepss").is_dir()), None)
if RAIZ is None:
    raise RuntimeError("No encuentro src/hepss: corre el notebook desde dentro del repositorio")
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
os.chdir(RAIZ)  # rutas relativas (configs/, data/, plots/) respecto a la raíz
Path("plots").mkdir(exist_ok=True)

# Recarga automática de hepss: si editas src/hepss/*.py, no hace falta reiniciar el kernel
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Para forzar los datos sintéticos (sin red), descomentar:
# os.environ["HEPSS_SINTETICO"] = "1"
print("Colab:", EN_COLAB, "| raíz del repositorio:", RAIZ)

### Datos: CMS Open Data o respaldo sintético

`hepss.io.abrir_events` intenta leer CMS Open Data por HTTPS (y por XRootD si `fsspec-xrootd` está instalado).
Si no lo consigue en 30 s (sin red, servidor caído, firewall) o si la ruta no está configurada, **genera una vez
un archivo sintético en `data/` con las mismas ramas** y lo usa, avisando con un `UserWarning`. La variable
`ORIGEN` vale `"opendata"` o `"sintetico"`. Usa siempre `etiqueta_cms(ax, ORIGEN)` en tus gráficas: con datos
sintéticos pone *Simulación sintética* en lugar de *CMS Open Data*, para que nunca se confundan.

Las verificaciones están escritas para pasar con ambos orígenes. Las que solo tienen sentido con datos reales
se comprueban solo si `ORIGEN == "opendata"`.

In [ ]:
import time
import numpy as np
import awkward as ak
import hist
import matplotlib.pyplot as plt
import mplhep as hep

from hepss.io import abrir_events, RAMAS_MUON
from hepss.graficas import etiqueta_cms

N_EVENTOS = 2_000_000

## E2.1 Abrir el archivo y listar ramas

`abrir_events` hace el `uproot.open(...)["Events"]` por ti (con el respaldo sintético). Lista las ramas y sus tipos
con `.show()` e imprime el número de eventos (`num_entries`).

In [ ]:
arbol, ORIGEN = abrir_events("dimuon2012")
print("Origen de los datos:", ORIGEN)

# TODO: muestra las ramas con sus tipos y el número de eventos del árbol

In [ ]:
# Verificación E2.1
assert set(RAMAS_MUON) <= set(arbol.keys()), "Faltan ramas de muones"
if ORIGEN == "opendata":
    assert 61.0e6 < arbol.num_entries < 62.0e6

¿Qué significa el tipo `float[]` de `Muon_pt`? ¿En qué unidades están pT y φ?

**Tu respuesta:** _(escribe aquí)_

## E2.2 Leer las ramas a un `ak.Array`

Lee solo las ramas de `RAMAS_MUON` y solo los primeros `N_EVENTOS` en la variable `eventos`. Mide el tiempo de lectura
en `t_lectura` (segundos, con `time.perf_counter()`), que se usa en E2.6.

In [ ]:
# TODO
eventos = ...
t_lectura = ...

In [ ]:
# Verificación E2.2
assert ak.all(ak.num(eventos.Muon_pt) == eventos.nMuon)

## E2.3 Número de muones por evento

Histograma `h_n` de `nMuon` con `hist` (eje `hist.axis.Integer`), graficado con `mplhep.histplot` en escala log y con
`etiqueta_cms(ax, ORIGEN)`. Guarda la figura en `plots/nmuon.png`. Calcula `frac_2omas`, la fracción de eventos con
2 o más muones.

In [ ]:
# TODO
h_n = ...
frac_2omas = ...

In [ ]:
# Verificación E2.3
print(f"Fracción con 2 o más muones: {frac_2omas:.1%}")
assert frac_2omas > 0.5

El dataset viene de un trigger de doble muon. ¿Por qué aun así hay eventos con un solo muon?

**Tu respuesta:** _(escribe aquí)_

## E2.4 pT, η y φ de todos los muones

Aplana con `ak.flatten` para pasar de "lista de muones por evento" a "lista de todos los muones" (arreglos de numpy
`pt_todos`, `eta_todos`, `phi_todos`). Haz `h_pt` (0 a 100 GeV, escala log en y), `h_eta` y `h_phi`, en una figura de
tres paneles guardada en `plots/pt_eta_phi.png`.

In [ ]:
# TODO
pt_todos = ...
eta_todos = ...
phi_todos = ...
h_pt = ...
h_eta = ...
h_phi = ...

In [ ]:
# Verificación E2.4
assert len(pt_todos) == ak.sum(eventos.nMuon)
assert h_pt.sum(flow=True) == len(pt_todos)

## E2.5 Muon de mayor pT de cada evento

Usa `ak.argmax(..., axis=1, keepdims=True)` para obtener el muon líder de cada evento (`pt_lider`, arreglo de numpy).
Cuidado con los eventos sin muones. Compara las **formas** de `h_pt` y del espectro del líder en la misma gráfica,
ambas normalizadas a 1 (`density=True`), y calcula la fracción con pT > 20 GeV en cada caso (`f_todos`, `f_lider`).

In [ ]:
# TODO
pt_lider = ...
f_todos = ...
f_lider = ...

In [ ]:
# Verificación E2.5
print(f"Fracción con pT > 20 GeV: todos {f_todos:.3%}, líder {f_lider:.3%}")
assert f_lider > f_todos
assert len(pt_lider) == ak.sum(eventos.nMuon > 0)

¿Por qué la comparación solo tiene sentido con las distribuciones normalizadas? ¿Qué daría `eventos.Muon_pt[:, 0]` y en qué caso falla?

**Tu respuesta:** _(escribe aquí)_

## E2.6 Tiempo, memoria y lectura por bloques

Reporta el tiempo de E2.2, la memoria del `ak.Array` (`eventos.nbytes`) y la memoria total del proceso (`psutil`).
Luego llena un histograma `h_bloques` con el mismo eje que `h_pt` leyendo por bloques con `arbol.iterate(...,
step_size="50 MB", entry_stop=N_EVENTOS)`.

In [ ]:
import psutil
proc = psutil.Process(os.getpid())

# TODO
h_bloques = ...

In [ ]:
# Verificación E2.6
assert np.array_equal(h_bloques.values(flow=True), h_pt.values(flow=True))

¿Por qué la verificación usa `flow=True`? ¿Por qué la memoria del proceso es mayor que `eventos.nbytes`? ¿Cabe el archivo completo en la memoria de Colab?

**Tu respuesta:** _(escribe aquí)_

## E2.7 Funciones en `src/hepss/`

Ya repetiste código de lectura y de gráficas. Implementa en el paquete (los esqueletos y su especificación ya están ahí):

* `hepss.io.leer_muones(ruta, n_eventos)` en `src/hepss/io.py`,
* `hepss.graficas.hist_cms(h, ...)` en `src/hepss/graficas.py`.

Después cambia las celdas de arriba para que usen estas funciones. Los notebooks 02 y 03 las necesitan.

In [ ]:
# Verificación E2.7
import inspect
from hepss.io import leer_muones
from hepss.graficas import hist_cms

eventos2, origen2 = leer_muones("dimuon2012", n_eventos=1000)
assert len(eventos2) == 1000 and origen2 in ("opendata", "sintetico")
ax = hist_cms(hist.Hist(h_eta.axes[0]).fill(ak.to_numpy(ak.flatten(eventos2.Muon_eta))), origen=origen2)
assert ax is not None
plt.close()
assert leer_muones.__module__ == "hepss.io" and hist_cms.__module__ == "hepss.graficas"
print(inspect.signature(leer_muones))
print(inspect.signature(hist_cms))

## Pregunta para la bitácora

¿Por qué la distribución en η no es plana y la de φ sí?

**Tu respuesta:** _(escribe aquí)_